# XGBoost robusto con validación cruzada y métricas completas
Este notebook replica el flujo robusto de tu modelo logístico, pero usando XGBoost, con balanceo, validación cruzada y reportes completos.

In [3]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, accuracy_score, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [4]:
# === CONFIGURACIÓN ===
INPUT_PATH = r'c:/Users/Gonzalo/Documents/GitHub/TesisAustral/Archivos_tesis/intermediate/selected/04_2025_08_23'
BASE_OUTPUT_PATH = r'c:/Users/Gonzalo/Documents/GitHub/TesisAustral/Archivos_tesis/models'
MODEL_NAME = 'XGBoost'
INTENTO = 2  # Cambia según versión
fecha_actual = datetime.today().strftime('%Y-%m-%d')
OUTPUT_PATH = os.path.join(BASE_OUTPUT_PATH, f'{MODEL_NAME}_{INTENTO:02d}_{fecha_actual}')
os.makedirs(OUTPUT_PATH, exist_ok=True)

BALANCE_METHODS = {
    'NONE': None,
    'SMOTE': SMOTE(random_state=42),
    'UNDER': RandomUnderSampler(random_state=42),
    'SMOTEENN': SMOTEENN(random_state=42)
}

metricas_totales = []

# Detectar variantes de datasets
variantes_X = sorted([f for f in os.listdir(INPUT_PATH) if f.startswith('X_train')])

In [ ]:
for balance_name, balancer in BALANCE_METHODS.items():
    print(f"=== Balanceo: {balance_name} ===")
    output_subdir = os.path.join(OUTPUT_PATH, balance_name)
    os.makedirs(output_subdir, exist_ok=True)

    for x_file in variantes_X:
        base_name = x_file.replace('X_train_', '').replace('.parquet', '')
        try:
            print(f'🔍 Procesando: {base_name}')

            # Cargar datos
            X_train = pd.read_parquet(os.path.join(INPUT_PATH, f'X_train_{base_name}.parquet'))
            X_test = pd.read_parquet(os.path.join(INPUT_PATH, f'X_test_{base_name}.parquet'))
            y_train = pd.read_parquet(os.path.join(INPUT_PATH, f'y_train_{base_name}.parquet')).squeeze()
            y_test = pd.read_parquet(os.path.join(INPUT_PATH, f'y_test_{base_name}.parquet')).squeeze()

            # Balanceo solo en train
            if balancer is not None:
                X_train_bal, y_train_bal = balancer.fit_resample(X_train, y_train)
            else:
                X_train_bal, y_train_bal = X_train, y_train

            # Validación cruzada sobre el train balanceado
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            model_cv = XGBClassifier(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                use_label_encoder=False,
                eval_metric='mlogloss',
                random_state=42,
                n_jobs=-1
            )
            cv_scores = cross_val_score(model_cv, X_train_bal, y_train_bal, cv=cv, scoring='f1_macro')
            cv_f1_mean = np.mean(cv_scores)
            cv_f1_std = np.std(cv_scores)

            # Entrenamiento
            start = time.time()
            model = XGBClassifier(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                use_label_encoder=False,
                eval_metric='mlogloss',
                random_state=42,
                n_jobs=-1
            )
            model.fit(X_train_bal, y_train_bal)
            tiempo = (time.time() - start) / 60

            # Predicción
            y_pred_train = model.predict(X_train_bal)
            y_pred_test = model.predict(X_test)
            y_proba = model.predict_proba(X_test)

            # Reporte completo
            report_test = pd.DataFrame(classification_report(y_test, y_pred_test, output_dict=True)).T
            report_train = pd.DataFrame(classification_report(y_train_bal, y_pred_train, output_dict=True)).T
            report_test['set'] = 'test'
            report_train['set'] = 'train'
            full_report = pd.concat([report_train, report_test])
            full_report['tiempo_min'] = tiempo

            # Guardar CSV de métricas por clase
            full_report.to_csv(os.path.join(output_subdir, f'metricas_{base_name}.csv'))

            # Matriz de confusión
            cm = confusion_matrix(y_test, y_pred_test)

            # Guardar PDF con visualizaciones
            with PdfPages(os.path.join(output_subdir, f'reporte_{base_name}.pdf')) as pdf:
                # Confusion Matrix
                ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap='Blues')
                plt.title('Matriz de Confusión')
                pdf.savefig(); plt.close()

                # ROC por clase
                classes = np.unique(y_test)
                y_bin = label_binarize(y_test, classes=classes)
                plt.figure()
                for i, clase in enumerate(classes):
                    fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
                    roc_auc = auc(fpr, tpr)
                    plt.plot(fpr, tpr, label=f'Clase {clase} (AUC={roc_auc:.2f})')
                plt.plot([0, 1], [0, 1], 'k--')
                plt.title('Curvas ROC por clase')
                plt.xlabel('FPR')
                plt.ylabel('TPR')
                plt.legend()
                pdf.savefig(); plt.close()

                # PR Curve por clase
                plt.figure()
                for i, clase in enumerate(classes):
                    precision, recall, _ = precision_recall_curve(y_bin[:, i], y_proba[:, i])
                    pr_auc = average_precision_score(y_bin[:, i], y_proba[:, i])
                    plt.plot(recall, precision, label=f'Clase {clase} (PR AUC={pr_auc:.2f})')
                plt.title('Curvas Precision-Recall por clase')
                plt.xlabel('Recall')
                plt.ylabel('Precision')
                plt.legend()
                pdf.savefig(); plt.close()

            # Resumen final para comparabilidad
            resumen = {
                'modelo': base_name,
                'balanceo': balance_name,
                'accuracy_test': accuracy_score(y_test, y_pred_test),
                'macro_f1_test': report_test.loc['macro avg', 'f1-score'],
                'weighted_f1_test': report_test.loc['weighted avg', 'f1-score'],
                'accuracy_train': accuracy_score(y_train_bal, y_pred_train),
                'macro_f1_train': report_train.loc['macro avg', 'f1-score'],
                'weighted_f1_train': report_train.loc['weighted avg', 'f1-score'],
                'tiempo_min': tiempo,
                'sobreajuste_macro_f1': report_train.loc['macro avg', 'f1-score'] - report_test.loc['macro avg', 'f1-score'],
                'cv_macro_f1_mean': cv_f1_mean,
                'cv_macro_f1_std': cv_f1_std
            }
            for clase in report_test.index:
                if clase not in ['accuracy', 'macro avg', 'weighted avg']:
                    resumen[f'f1_{clase}_test'] = report_test.loc[clase, 'f1-score']
                    resumen[f'recall_{clase}_test'] = report_test.loc[clase, 'recall']
                    resumen[f'precision_{clase}_test'] = report_test.loc[clase, 'precision']
                    resumen[f'f1_{clase}_train'] = report_train.loc[clase, 'f1-score']
                    resumen[f'recall_{clase}_train'] = report_train.loc[clase, 'recall']
                    resumen[f'precision_{clase}_train'] = report_train.loc[clase, 'precision']
            metricas_totales.append(resumen)

            print(f"✅ {base_name} [{balance_name}]: F1_clase_1_test={resumen.get('f1_1_test', 0):.3f}, Recall_clase_1_test={resumen.get('recall_1_test', 0):.3f}, Tiempo={tiempo:.2f}min")

        except Exception as e:
            print(f'❌ Error en {base_name} con balanceo {balance_name}: {e}')

# Guardar resumen final unificado
df_final = pd.DataFrame(metricas_totales)
df_final.to_csv(os.path.join(OUTPUT_PATH, 'resumen_modelos_xgboost.csv'), index=False)
print(f"\n🏆 Resumen de todas las variantes guardado en: {os.path.join(OUTPUT_PATH, 'resumen_modelos_xgboost.csv')}")


SyntaxError: unterminated f-string literal (detected at line 2) (681651914.py, line 2)

: 

# XGBoost robusto con validación cruzada y métricas completas
Este notebook replica el flujo robusto de tu modelo logístico, pero usando XGBoost, con balanceo, validación cruzada y reportes completos.

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, accuracy_score, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [ ]:
# === CONFIGURACIÓN ===
INPUT_PATH = r'"C:\Use
BASE_OUTPUT_PATH = r'./Archivos_tesis/models'
MODEL_NAME = 'XGBoost'
INTENTO = 1  # Cambia según versión
fecha_actual = datetime.today().strftime('%Y-%m-%d')
OUTPUT_PATH = os.path.join(BASE_OUTPUT_PATH, f'{MODEL_NAME}_{INTENTO:02d}_{fecha_actual}')
os.makedirs(OUTPUT_PATH, exist_ok=True)

BALANCE_METHODS = {
    'NONE': None,
    'SMOTE': SMOTE(random_state=42),
    'UNDER': RandomUnderSampler(random_state=42),
    'SMOTEENN': SMOTEENN(random_state=42)
}

metricas_totales = []

# Detectar variantes de datasets
variantes_X = sorted([f for f in os.listdir(INPUT_PATH) if f.startswith('X_train')])

FileNotFoundError: [WinError 3] El sistema no puede encontrar la ruta especificada: './Archivos_tesis/intermediate/selected/04_2025_08_23'

In [ ]:
for balance_name, balancer in BALANCE_METHODS.items():
    print(f'
=== Balanceo: {balance_name} ===')
    output_subdir = os.path.join(OUTPUT_PATH, balance_name)
    os.makedirs(output_subdir, exist_ok=True)

    for x_file in variantes_X:
        base_name = x_file.replace('X_train_', '').replace('.parquet', '')
        try:
            print(f'🔍 Procesando: {base_name}')

            # Cargar datos
            X_train = pd.read_parquet(os.path.join(INPUT_PATH, f'X_train_{base_name}.parquet'))
            X_test = pd.read_parquet(os.path.join(INPUT_PATH, f'X_test_{base_name}.parquet'))
            y_train = pd.read_parquet(os.path.join(INPUT_PATH, f'y_train_{base_name}.parquet')).squeeze()
            y_test = pd.read_parquet(os.path.join(INPUT_PATH, f'y_test_{base_name}.parquet')).squeeze()

            # Balanceo solo en train
            if balancer is not None:
                X_train_bal, y_train_bal = balancer.fit_resample(X_train, y_train)
            else:
                X_train_bal, y_train_bal = X_train, y_train

            # Validación cruzada sobre el train balanceado
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            model_cv = XGBClassifier(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                use_label_encoder=False,
                eval_metric='mlogloss',
                random_state=42,
                n_jobs=-1
            )
            cv_scores = cross_val_score(model_cv, X_train_bal, y_train_bal, cv=cv, scoring='f1_macro')
            cv_f1_mean = np.mean(cv_scores)
            cv_f1_std = np.std(cv_scores)

            # Entrenamiento
            start = time.time()
            model = XGBClassifier(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                use_label_encoder=False,
                eval_metric='mlogloss',
                random_state=42,
                n_jobs=-1
            )
            model.fit(X_train_bal, y_train_bal)
            tiempo = (time.time() - start) / 60

            # Predicción
            y_pred_train = model.predict(X_train_bal)
            y_pred_test = model.predict(X_test)
            y_proba = model.predict_proba(X_test)

            # Reporte completo
            report_test = pd.DataFrame(classification_report(y_test, y_pred_test, output_dict=True)).T
            report_train = pd.DataFrame(classification_report(y_train_bal, y_pred_train, output_dict=True)).T
            report_test['set'] = 'test'
            report_train['set'] = 'train'
            full_report = pd.concat([report_train, report_test])
            full_report['tiempo_min'] = tiempo

            # Guardar CSV de métricas por clase
            full_report.to_csv(os.path.join(output_subdir, f'metricas_{base_name}.csv'))

            # Matriz de confusión
            cm = confusion_matrix(y_test, y_pred_test)

            # Guardar PDF con visualizaciones
            with PdfPages(os.path.join(output_subdir, f'reporte_{base_name}.pdf')) as pdf:
                # Confusion Matrix
                ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap='Blues')
                plt.title('Matriz de Confusión')
                pdf.savefig(); plt.close()

                # ROC por clase
                classes = np.unique(y_test)
                y_bin = label_binarize(y_test, classes=classes)
                plt.figure()
                for i, clase in enumerate(classes):
                    fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
                    roc_auc = auc(fpr, tpr)
                    plt.plot(fpr, tpr, label=f'Clase {clase} (AUC={roc_auc:.2f})')
                plt.plot([0, 1], [0, 1], 'k--')
                plt.title('Curvas ROC por clase')
                plt.xlabel('FPR')
                plt.ylabel('TPR')
                plt.legend()
                pdf.savefig(); plt.close()

                # PR Curve por clase
                plt.figure()
                for i, clase in enumerate(classes):
                    precision, recall, _ = precision_recall_curve(y_bin[:, i], y_proba[:, i])
                    pr_auc = average_precision_score(y_bin[:, i], y_proba[:, i])
                    plt.plot(recall, precision, label=f'Clase {clase} (PR AUC={pr_auc:.2f})')
                plt.title('Curvas Precision-Recall por clase')
                plt.xlabel('Recall')
                plt.ylabel('Precision')
                plt.legend()
                pdf.savefig(); plt.close()

            # Resumen final para comparabilidad
            resumen = {
                'modelo': base_name,
                'balanceo': balance_name,
                'accuracy_test': accuracy_score(y_test, y_pred_test),
                'macro_f1_test': report_test.loc['macro avg', 'f1-score'],
                'weighted_f1_test': report_test.loc['weighted avg', 'f1-score'],
                'accuracy_train': accuracy_score(y_train_bal, y_pred_train),
                'macro_f1_train': report_train.loc['macro avg', 'f1-score'],
                'weighted_f1_train': report_train.loc['weighted avg', 'f1-score'],
                'tiempo_min': tiempo,
                'sobreajuste_macro_f1': report_train.loc['macro avg', 'f1-score'] - report_test.loc['macro avg', 'f1-score'],
                'cv_macro_f1_mean': cv_f1_mean,
                'cv_macro_f1_std': cv_f1_std
            }
            for clase in report_test.index:
                if clase not in ['accuracy', 'macro avg', 'weighted avg']:
                    resumen[f'f1_{clase}_test'] = report_test.loc[clase, 'f1-score']
                    resumen[f'recall_{clase}_test'] = report_test.loc[clase, 'recall']
                    resumen[f'precision_{clase}_test'] = report_test.loc[clase, 'precision']
                    resumen[f'f1_{clase}_train'] = report_train.loc[clase, 'f1-score']
                    resumen[f'recall_{clase}_train'] = report_train.loc[clase, 'recall']
                    resumen[f'precision_{clase}_train'] = report_train.loc[clase, 'precision']
            metricas_totales.append(resumen)

            print(f'✅ {base_name} [{balance_name}]: F1_clase_1_test={resumen.get('f1_1_test', 0):.3f}, Recall_clase_1_test={resumen.get('recall_1_test', 0):.3f}, Tiempo={tiempo:.2f}min')

        except Exception as e:
            print(f'❌ Error en {base_name} con balanceo {balance_name}: {e}')

# Guardar resumen final unificado
df_final = pd.DataFrame(metricas_totales)
df_final.to_csv(os.path.join(OUTPUT_PATH, 'resumen_modelos_xgboost.csv'), index=False)
print(f'
🏆 Resumen de todas las variantes guardado en: {os.path.join(OUTPUT_PATH, 'resumen_modelos_xgboost.csv')}')